# Performance Analytics — Mutual Fund Risk & Return Metrics
**Project:** Bluestock Mutual Fund Analytics | **Day 4:** Performance Analytics  
**Covers:** Daily returns, CAGR, Sharpe, Sortino, Alpha/Beta, Max Drawdown, Fund Scorecard, Benchmark comparison


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings, os

warnings.filterwarnings('ignore')

PROC = '../data/processed'
REP  = '../reports'
os.makedirs(REP, exist_ok=True)

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

RF_RATE = 0.065   # RBI repo rate proxy, 6.5%
TRADING_DAYS = 252

print('Libraries loaded')

Libraries loaded


## Load Data

In [2]:
fm   = pd.read_csv(f'{PROC}/01_fund_master.csv')
nav  = pd.read_csv(f'{PROC}/02_nav_history.csv', parse_dates=['date'])
perf = pd.read_csv(f'{PROC}/07_scheme_performance.csv')
bi   = pd.read_csv(f'{PROC}/10_benchmark_indices.csv', parse_dates=['date'])

nav = nav.merge(fm[['amfi_code','scheme_name','fund_house','sub_category','plan']], on='amfi_code', how='left')

# Pivot NAV to wide format: dates as rows, funds as columns
nav_wide = nav.pivot_table(index='date', columns='amfi_code', values='nav')
nav_wide = nav_wide.sort_index()

print(f'NAV wide shape : {nav_wide.shape}')
print(f'Date range     : {nav_wide.index.min().date()} to {nav_wide.index.max().date()}')
print(f'Schemes        : {nav_wide.shape[1]}')

NAV wide shape : (1608, 40)
Date range     : 2022-01-03 to 2026-05-29
Schemes        : 40


---
## Step 1 — Compute Daily Returns
`daily_return = (NAV_t / NAV_t-1) - 1` for all 40 schemes.


In [3]:
daily_returns = nav_wide.pct_change().dropna(how='all')

print(f'Daily returns shape : {daily_returns.shape}')
print(f'Mean daily return    : {daily_returns.mean().mean()*100:.4f}%')
print(f'Std daily return     : {daily_returns.std().mean()*100:.4f}%')
print(f'Min daily return     : {daily_returns.min().min()*100:.2f}%')
print(f'Max daily return     : {daily_returns.max().max()*100:.2f}%')

# Validate distribution
fig, ax = plt.subplots(figsize=(12, 5))
all_returns = daily_returns.values.flatten()
all_returns = all_returns[~np.isnan(all_returns)]
ax.hist(all_returns*100, bins=100, color='#2196F3', alpha=0.75, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_title('Distribution of Daily Returns — All 40 Schemes\n(Validation: should be roughly bell-shaped, centered near 0)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(f'{REP}/chart_16_daily_return_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('Distribution check:')
print(f'  Skewness : {stats.skew(all_returns):.3f}  (near 0 = symmetric)')
print(f'  Kurtosis : {stats.kurtosis(all_returns):.3f}  (near 0 = normal-like tails)')
print('  PASS: Distribution looks reasonable, no extreme data errors' if abs(stats.skew(all_returns)) < 2 else '  WARNING: Check for outliers')

Daily returns shape : (1607, 40)
Mean daily return    : 0.0451%
Std daily return     : 0.7967%
Min daily return     : -5.81%
Max daily return     : 6.47%

Distribution check:
  Skewness : 0.112  (near 0 = symmetric)
  Kurtosis : 3.101  (near 0 = normal-like tails)
  PASS: Distribution looks reasonable, no extreme data errors


---
## Step 2 — Compute CAGR (1yr, 3yr, 5yr)
`CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`

Note: Data spans Jan 2022 to May 2026 (~4.4 years), so 5yr CAGR uses maximum available history where less than 5 years exist.


In [4]:
def compute_cagr(series, years):
    series = series.dropna()
    if len(series) < 2:
        return np.nan
    end_date   = series.index.max()
    start_date = end_date - pd.DateOffset(years=years)
    series_window = series[series.index >= start_date]
    if len(series_window) < 2:
        return np.nan
    nav_start = series_window.iloc[0]
    nav_end   = series_window.iloc[-1]
    actual_years = (series_window.index[-1] - series_window.index[0]).days / 365.25
    if actual_years <= 0 or nav_start <= 0:
        return np.nan
    cagr = (nav_end / nav_start) ** (1/actual_years) - 1
    return cagr * 100

cagr_rows = []
for code_id in nav_wide.columns:
    series = nav_wide[code_id]
    cagr_rows.append({
        'amfi_code'  : code_id,
        'cagr_1yr_pct': compute_cagr(series, 1),
        'cagr_3yr_pct': compute_cagr(series, 3),
        'cagr_5yr_pct': compute_cagr(series, 5),
    })

cagr_df = pd.DataFrame(cagr_rows).merge(
    fm[['amfi_code','scheme_name','fund_house','sub_category','plan']],
    on='amfi_code', how='left'
)
cagr_df = cagr_df[['amfi_code','scheme_name','fund_house','sub_category','plan',
                    'cagr_1yr_pct','cagr_3yr_pct','cagr_5yr_pct']]
cagr_df = cagr_df.sort_values('cagr_3yr_pct', ascending=False).reset_index(drop=True)

print('CAGR Comparison Table (Top 10 by 3yr CAGR):')
print(cagr_df.head(10).to_string(index=False))

CAGR Comparison Table (Top 10 by 3yr CAGR):
 amfi_code                                        scheme_name               fund_house sub_category    plan  cagr_1yr_pct  cagr_3yr_pct  cagr_5yr_pct
    119094                Axis Midcap Fund - Regular - Growth         Axis Mutual Fund      Mid Cap Regular     22.277897     35.102528     28.214417
    148567      Mirae Asset Large Cap Fund - Regular - Growth           Mirae Asset MF    Large Cap Regular     20.375957     33.991970     30.974108
    120504          ICICI Pru Bluechip Fund - Direct - Growth      ICICI Prudential MF    Large Cap  Direct     13.073788     32.478927     23.295119
    100033 HDFC Mid-Cap Opportunities Fund - Regular - Growth         HDFC Mutual Fund      Mid Cap Regular     53.277195     32.433971     30.123153
    120505           ICICI Pru Midcap Fund - Regular - Growth      ICICI Prudential MF      Mid Cap Regular     29.627681     31.769243     32.827406
    119551          SBI Bluechip Fund - Regular Plan - G

---
## Step 3 — Sharpe Ratio
`Sharpe = (Rp - Rf) / Std(Rp) × √252`, Rf = 6.5%


In [7]:
sharpe_rows = []
for code_id in nav_wide.columns:
    ret = daily_returns[code_id].dropna()
    if len(ret) < 30:
        sharpe_rows.append({'amfi_code': code_id, 'sharpe_ratio': np.nan})
        continue
    ann_return = ret.mean() * TRADING_DAYS
    ann_std    = ret.std() * np.sqrt(TRADING_DAYS)
    sharpe = (ann_return - RF_RATE) / ann_std if ann_std > 0 else np.nan
    sharpe_rows.append({'amfi_code': code_id, 'sharpe_ratio': sharpe,
                        'ann_return_pct': ann_return*100, 'ann_std_pct': ann_std*100})

sharpe_df = pd.DataFrame(sharpe_rows).merge(
    fm[['amfi_code','scheme_name','fund_house']], on='amfi_code', how='left'
)
sharpe_df = sharpe_df.sort_values('sharpe_ratio', ascending=False).reset_index(drop=True)
sharpe_df['sharpe_rank'] = sharpe_df['sharpe_ratio'].rank(ascending=False)

print('Sharpe Ratio Ranking (Top 10):')
print(sharpe_df[['scheme_name','fund_house','sharpe_ratio','ann_return_pct','ann_std_pct']].head(10).to_string(index=False))

Sharpe Ratio Ranking (Top 10):
                                       scheme_name               fund_house  sharpe_ratio  ann_return_pct  ann_std_pct
     Mirae Asset Large Cap Fund - Regular - Growth           Mirae Asset MF      1.068224       19.345399    12.025007
            Kotak Flexicap Fund - Regular - Growth        Kotak Mahindra MF      0.965561       19.490975    13.454334
     Mirae Asset Tax Saver Fund - Regular - Growth           Mirae Asset MF      0.919047       20.253117    14.964549
          ICICI Pru Midcap Fund - Regular - Growth      ICICI Prudential MF      0.883256       20.924579    16.331149
         SBI Bluechip Fund - Regular Plan - Growth          SBI Mutual Fund      0.860977       16.518799    11.636542
                DSP Midcap Fund - Regular - Growth          DSP Mutual Fund      0.832885       19.012350    15.022898
HDFC Mid-Cap Opportunities Fund - Regular - Growth         HDFC Mutual Fund      0.808268       19.455821    16.029119
    Nippon India 

---
## Step 4 — Sortino Ratio
Same as Sharpe but denominator uses only downside deviation (negative return days only).


In [8]:
sortino_rows = []
for code_id in nav_wide.columns:
    ret = daily_returns[code_id].dropna()
    if len(ret) < 30:
        sortino_rows.append({'amfi_code': code_id, 'sortino_ratio': np.nan})
        continue
    ann_return = ret.mean() * TRADING_DAYS
    downside = ret[ret < 0]
    downside_std = downside.std() * np.sqrt(TRADING_DAYS) if len(downside) > 0 else np.nan
    sortino = (ann_return - RF_RATE) / downside_std if downside_std and downside_std > 0 else np.nan
    sortino_rows.append({'amfi_code': code_id, 'sortino_ratio': sortino})

sortino_df = pd.DataFrame(sortino_rows).merge(
    fm[['amfi_code','scheme_name']], on='amfi_code', how='left'
)
sortino_df = sortino_df.sort_values('sortino_ratio', ascending=False).reset_index(drop=True)
sortino_df['sortino_rank'] = sortino_df['sortino_ratio'].rank(ascending=False)

print('Sortino Ratio Ranking (Top 10):')
print(sortino_df.head(10).to_string(index=False))

Sortino Ratio Ranking (Top 10):
 amfi_code  sortino_ratio                                        scheme_name  sortino_rank
    148567       1.490739      Mirae Asset Large Cap Fund - Regular - Growth           1.0
    120843       1.479503             Kotak Flexicap Fund - Regular - Growth           2.0
    148569       1.352815      Mirae Asset Tax Saver Fund - Regular - Growth           3.0
    119551       1.291483          SBI Bluechip Fund - Regular Plan - Growth           4.0
    120505       1.285843           ICICI Pru Midcap Fund - Regular - Growth           5.0
    149323       1.167793                 DSP Midcap Fund - Regular - Growth           6.0
    100033       1.144216 HDFC Mid-Cap Opportunities Fund - Regular - Growth           7.0
    118632       1.098880     Nippon India Large Cap Fund - Regular - Growth           8.0
    119598       1.067256         SBI Small Cap Fund - Regular Plan - Growth           9.0
    120504       1.063964          ICICI Pru Bluechip Fund

---
## Step 5 — Alpha and Beta (CAPM Regression)
OLS regression of fund returns on NIFTY100 returns using `scipy.stats.linregress`.  
Alpha = intercept × 252 (annualized)


In [9]:
# Benchmark: NIFTY100 daily returns
nifty100 = bi[bi['index_name']=='NIFTY100'].set_index('date')['close_value'].sort_index()
nifty100_ret = nifty100.pct_change().dropna()

alpha_beta_rows = []
for code_id in nav_wide.columns:
    fund_ret = daily_returns[code_id].dropna()

    # Align dates
    common_idx = fund_ret.index.intersection(nifty100_ret.index)
    if len(common_idx) < 30:
        alpha_beta_rows.append({'amfi_code': code_id, 'alpha_pct': np.nan, 'beta': np.nan,
                                'r_squared': np.nan, 'p_value': np.nan})
        continue

    x = nifty100_ret.loc[common_idx].values
    y = fund_ret.loc[common_idx].values

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    alpha_annualized = intercept * TRADING_DAYS * 100   # as %
    beta = slope

    alpha_beta_rows.append({
        'amfi_code': code_id,
        'alpha_pct': alpha_annualized,
        'beta': beta,
        'r_squared': r_value**2,
        'p_value': p_value,
        'std_err': std_err
    })

alpha_beta_df = pd.DataFrame(alpha_beta_rows).merge(
    fm[['amfi_code','scheme_name','fund_house','sub_category','plan']],
    on='amfi_code', how='left'
)
alpha_beta_df = alpha_beta_df[['amfi_code','scheme_name','fund_house','sub_category','plan',
                                'alpha_pct','beta','r_squared','p_value']]
alpha_beta_df = alpha_beta_df.sort_values('alpha_pct', ascending=False).reset_index(drop=True)

print('Alpha & Beta vs NIFTY100 (Top 10 by Alpha):')
print(alpha_beta_df.head(10).to_string(index=False))

# Save deliverable
alpha_beta_df.to_csv('../alpha_beta.csv', index=False)
print()
print('Saved -> alpha_beta.csv')

Alpha & Beta vs NIFTY100 (Top 10 by Alpha):
 amfi_code                                        scheme_name          fund_house sub_category    plan  alpha_pct      beta    r_squared  p_value
    119598         SBI Small Cap Fund - Regular Plan - Growth     SBI Mutual Fund    Small Cap Regular  30.336965 -0.023196 1.414258e-04 0.687179
    149324              DSP Small Cap Fund - Regular - Growth     DSP Mutual Fund    Small Cap Regular  30.057878  0.011455 3.532991e-05 0.840494
    120505           ICICI Pru Midcap Fund - Regular - Growth ICICI Prudential MF      Mid Cap Regular  29.263583  0.000549 1.345534e-07 0.990090
    148569      Mirae Asset Tax Saver Fund - Regular - Growth      Mirae Asset MF         ELSS Regular  28.270368  0.018134 1.748889e-04 0.654295
    120843             Kotak Flexicap Fund - Regular - Growth   Kotak Mahindra MF    Flexi Cap Regular  27.330465 -0.022830 3.430543e-04 0.530528
    100033 HDFC Mid-Cap Opportunities Fund - Regular - Growth    HDFC Mutual Fun

---
## Step 6 — Maximum Drawdown
`Drawdown = NAV / running_max(NAV) - 1`. Find worst drawdown date range per fund.


In [10]:
def compute_max_drawdown(series):
    series = series.dropna()
    if len(series) < 2:
        return np.nan, None, None
    running_max = series.cummax()
    drawdown = series / running_max - 1
    max_dd = drawdown.min()
    trough_date = drawdown.idxmin()
    # Find the peak before the trough
    peak_date = series[series.index <= trough_date].idxmax()
    return max_dd * 100, peak_date, trough_date

dd_rows = []
for code_id in nav_wide.columns:
    series = nav_wide[code_id]
    max_dd, peak_dt, trough_dt = compute_max_drawdown(series)
    dd_rows.append({
        'amfi_code': code_id,
        'max_drawdown_pct': max_dd,
        'peak_date': peak_dt,
        'trough_date': trough_dt
    })

dd_df = pd.DataFrame(dd_rows).merge(
    fm[['amfi_code','scheme_name','fund_house']], on='amfi_code', how='left'
)
dd_df = dd_df.sort_values('max_drawdown_pct', ascending=True).reset_index(drop=True)
dd_df['dd_rank'] = dd_df['max_drawdown_pct'].rank(ascending=False)  # smaller DD = better = higher rank

print('Worst 10 Drawdowns:')
print(dd_df[['scheme_name','fund_house','max_drawdown_pct','peak_date','trough_date']].head(10).to_string(index=False))

Worst 10 Drawdowns:
                                   scheme_name               fund_house  max_drawdown_pct  peak_date trough_date
     SBI Small Cap Fund - Direct Plan - Growth          SBI Mutual Fund        -52.574221 2023-01-17  2025-10-28
        Axis Small Cap Fund - Regular - Growth         Axis Mutual Fund        -51.677754 2025-05-22  2026-05-11
        ABSL Small Cap Fund - Regular - Growth Aditya Birla Sun Life MF        -35.446916 2024-11-21  2026-05-11
         DSP Small Cap Fund - Regular - Growth          DSP Mutual Fund        -31.171900 2024-05-03  2025-01-03
    SBI Small Cap Fund - Regular Plan - Growth          SBI Mutual Fund        -28.706006 2024-08-28  2025-05-14
           UTI Mid Cap Fund - Regular - Growth          UTI Mutual Fund        -28.001124 2025-01-07  2026-04-27
     HDFC Top 100 Fund - Regular Plan - Growth         HDFC Mutual Fund        -24.734441 2022-03-30  2022-09-15
 Kotak Emerging Equity Fund - Regular - Growth        Kotak Mahindra MF     

---
## Step 7 — Fund Scorecard (0–100)
Composite score:
`30% × 3yr return rank + 25% × Sharpe rank + 20% × Alpha rank + 15% × expense ratio rank (inverse) + 10% × max DD rank (inverse)`


In [11]:
scorecard = fm[['amfi_code','scheme_name','fund_house','sub_category','plan','expense_ratio_pct']].copy()

scorecard = scorecard.merge(cagr_df[['amfi_code','cagr_3yr_pct']], on='amfi_code', how='left')
scorecard = scorecard.merge(sharpe_df[['amfi_code','sharpe_ratio']], on='amfi_code', how='left')
scorecard = scorecard.merge(alpha_beta_df[['amfi_code','alpha_pct','beta']], on='amfi_code', how='left')
scorecard = scorecard.merge(dd_df[['amfi_code','max_drawdown_pct']], on='amfi_code', how='left')

# Convert each metric to a 0-100 percentile rank
scorecard['rank_3yr_return'] = scorecard['cagr_3yr_pct'].rank(pct=True) * 100
scorecard['rank_sharpe']     = scorecard['sharpe_ratio'].rank(pct=True) * 100
scorecard['rank_alpha']      = scorecard['alpha_pct'].rank(pct=True) * 100
# Expense ratio: LOWER is better, so invert
scorecard['rank_expense']    = (1 - scorecard['expense_ratio_pct'].rank(pct=True)) * 100
# Max drawdown: smaller (closer to 0) is better, so invert (less negative = better)
scorecard['rank_maxdd']      = scorecard['max_drawdown_pct'].rank(pct=True) * 100

scorecard['fund_score'] = (
    0.30 * scorecard['rank_3yr_return'] +
    0.25 * scorecard['rank_sharpe'] +
    0.20 * scorecard['rank_alpha'] +
    0.15 * scorecard['rank_expense'] +
    0.10 * scorecard['rank_maxdd']
).round(2)

scorecard = scorecard.sort_values('fund_score', ascending=False).reset_index(drop=True)
scorecard['overall_rank'] = range(1, len(scorecard)+1)

display_cols = ['overall_rank','scheme_name','fund_house','sub_category','plan',
                 'fund_score','cagr_3yr_pct','sharpe_ratio','alpha_pct',
                 'expense_ratio_pct','max_drawdown_pct']

print('Fund Scorecard — Top 10:')
print(scorecard[display_cols].head(10).to_string(index=False))

scorecard.to_csv('../fund_scorecard.csv', index=False)
print()
print('Saved -> fund_scorecard.csv')

Fund Scorecard — Top 10:
 overall_rank                                        scheme_name               fund_house sub_category    plan  fund_score  cagr_3yr_pct  sharpe_ratio  alpha_pct  expense_ratio_pct  max_drawdown_pct
            1      Mirae Asset Large Cap Fund - Regular - Growth           Mirae Asset MF    Large Cap Regular       85.88     33.991970      1.068224  26.983751               1.46        -11.265729
            2           ICICI Pru Midcap Fund - Regular - Growth      ICICI Prudential MF      Mid Cap Regular       82.50     31.769243      0.883256  29.263583               1.36        -18.188514
            3             Kotak Flexicap Fund - Regular - Growth        Kotak Mahindra MF    Flexi Cap Regular       81.62     29.575111      0.965561  27.330465               1.45        -12.973968
            4 HDFC Mid-Cap Opportunities Fund - Regular - Growth         HDFC Mutual Fund      Mid Cap Regular       80.38     32.433971      0.808268  27.195355               1.3

---
## Step 8 — Benchmark Comparison Chart
Top 5 funds (by scorecard) vs NIFTY50 and NIFTY100 over 3 years. Tracking error computed.


In [12]:
top5_codes = scorecard.head(5)['amfi_code'].tolist()

# 3-year window
end_date   = nav_wide.index.max()
start_date = end_date - pd.DateOffset(years=3)

nifty50  = bi[bi['index_name']=='NIFTY50'].set_index('date')['close_value'].sort_index()
nifty50  = nifty50[nifty50.index >= start_date]
nifty100_w = nifty100[nifty100.index >= start_date]

nifty50_norm  = (nifty50 / nifty50.iloc[0]) * 100
nifty100_norm = (nifty100_w / nifty100_w.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(16, 8))

tracking_errors = {}
colors_top5 = sns.color_palette('Set1', 5)

for i, code_id in enumerate(top5_codes):
    series = nav_wide[code_id]
    series_w = series[series.index >= start_date].dropna()
    if series_w.empty:
        continue
    series_norm = (series_w / series_w.iloc[0]) * 100
    name = fm[fm['amfi_code']==code_id]['scheme_name'].values[0][:30]
    ax.plot(series_norm.index, series_norm.values, linewidth=2,
            label=name, color=colors_top5[i])

    # Tracking error vs NIFTY100
    fund_ret_w = daily_returns[code_id][daily_returns[code_id].index >= start_date].dropna()
    common = fund_ret_w.index.intersection(nifty100_ret.index)
    if len(common) > 30:
        diff = fund_ret_w.loc[common] - nifty100_ret.loc[common]
        te = diff.std() * np.sqrt(TRADING_DAYS) * 100
        tracking_errors[name] = te

ax.plot(nifty50_norm.index, nifty50_norm.values, color='yellow',
        linewidth=2.5, linestyle='--', label='NIFTY50 (Benchmark)')
ax.plot(nifty100_norm.index, nifty100_norm.values, color='orange',
        linewidth=2.5, linestyle='--', label='NIFTY100 (Benchmark)')

ax.set_title('Top 5 Funds vs NIFTY50 & NIFTY100 — 3 Year Performance\n(Indexed to Base 100)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Indexed Value (Base = 100)')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig(f'{REP}/benchmark_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('Tracking Error vs NIFTY100 (3yr, annualized):')
for name, te in tracking_errors.items():
    print(f'  {name:<35} : {te:.2f}%')
print()
print('Chart saved -> reports/benchmark_comparison_chart.png')


Tracking Error vs NIFTY100 (3yr, annualized):
  Mirae Asset Large Cap Fund - R      : 18.79%
  ICICI Pru Midcap Fund - Regula      : 23.25%
  Kotak Flexicap Fund - Regular       : 20.64%
  HDFC Mid-Cap Opportunities Fun      : 22.48%
  ICICI Pru Bluechip Fund - Dire      : 18.73%

Chart saved -> reports/benchmark_comparison_chart.png


---
## Summary — Day 4 Deliverables


In [13]:
import os
print('='*60)
print('  Performance_Analytics.ipynb — COMPLETE')
print('='*60)
print()
print('Deliverables created:')
print(f'  fund_scorecard.csv          : {len(scorecard)} funds ranked')
print(f'  alpha_beta.csv               : {len(alpha_beta_df)} funds with alpha/beta')
print(f'  benchmark_comparison_chart.png : top 5 vs NIFTY50/100')
print(f'  chart_16_daily_return_dist.png : return distribution validation')
print()
print('Top 3 Funds by Scorecard:')
for i, row in scorecard.head(3).iterrows():
    print(f'  #{row["overall_rank"]:.0f}  {row["scheme_name"]:<35} Score: {row["fund_score"]:.1f}/100')
print()
print('Next:')
print('  git add .')
print('  git commit -m "Day 4: Performance analytics complete"')
print('  git push origin main')

  Performance_Analytics.ipynb — COMPLETE

Deliverables created:
  fund_scorecard.csv          : 40 funds ranked
  alpha_beta.csv               : 40 funds with alpha/beta
  benchmark_comparison_chart.png : top 5 vs NIFTY50/100
  chart_16_daily_return_dist.png : return distribution validation

Top 3 Funds by Scorecard:
  #1  Mirae Asset Large Cap Fund - Regular - Growth Score: 85.9/100
  #2  ICICI Pru Midcap Fund - Regular - Growth Score: 82.5/100
  #3  Kotak Flexicap Fund - Regular - Growth Score: 81.6/100

Next:
  git add .
  git commit -m "Day 4: Performance analytics complete"
  git push origin main
